# Exercise 1: Prompt Engineering Lab

In this exercise you will practice the core prompt-engineering techniques from Lecture 7:

1. Zero-shot vs few-shot prompting
2. Chain-of-Thought (CoT) reasoning
3. Role / persona prompting
4. Structured output with Pydantic
5. Evaluating prompt versions on a test set

**Prerequisites**: OpenAI API key configured, familiar with LangChain messages and tools.

## Setup

In [ ]:
!pip install langchain langchain-openai pydantic --quiet

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Part 1: Zero-shot vs Few-shot

We'll classify customer-support messages into: `billing`, `technical`, `sales`, `other`.

First a zero-shot prompt:

In [ ]:
zero_shot = ChatPromptTemplate.from_messages([
    ("system", "Classify the customer message into one of: billing, technical, sales, other. Respond with ONLY the label."),
    ("human", "{message}"),
])

test_messages = [
    ("My invoice shows a charge I didn't make", "billing"),
    ("The app crashes when I click export", "technical"),
    ("Do you offer discounts for 50 seats?", "sales"),
    ("Where can I find your office address?", "other"),
    ("I was charged twice this month", "billing"),
    ("My password reset email never arrived", "technical"),
]

def evaluate(prompt, cases):
    chain = prompt | llm
    correct = 0
    for msg, expected in cases:
        out = chain.invoke({"message": msg}).content.strip().lower()
        ok = out == expected
        correct += int(ok)
        print(f"{'OK ' if ok else 'BAD'} | expected={expected:<10} got={out:<20} | {msg}")
    print(f"\nAccuracy: {correct}/{len(cases)} = {correct/len(cases):.1%}")
    return correct / len(cases)

print("--- Zero-shot ---")
zero_acc = evaluate(zero_shot, test_messages)

**Your turn**: Add a few-shot version. Use 3-4 demonstrations covering at least 3 of the 4 classes.

In [ ]:
few_shot = ChatPromptTemplate.from_messages([
    ("system",
     "Classify the customer message into one of: billing, technical, sales, other. "
     "Respond with ONLY the label.\n\n"
     "Examples:\n"
     "Message: I think my credit card was charged wrong last month.\n"
     "Label: billing\n\n"
     "Message: The mobile app freezes on login.\n"
     "Label: technical\n\n"
     "Message: Can I get a quote for 200 licenses?\n"
     "Label: sales\n\n"
     "Message: What are your business hours?\n"
     "Label: other"),
    ("human", "{message}"),
])

print("--- Few-shot ---")
few_acc = evaluate(few_shot, test_messages)
print(f"\nDelta: {few_acc - zero_acc:+.1%}")

**TODO**: Add 3 trickier edge-case messages to `test_messages` (ambiguous or sarcastic) and re-run both prompts. Which version handles them better?

## Part 2: Chain-of-Thought

Consider this reasoning problem:

> A store sells apples for $0.50 each and oranges for $0.75 each. If I buy 6 apples and 4 oranges and pay with a $20 bill, how much change do I get?

In [ ]:
question = (
    "A store sells apples for $0.50 each and oranges for $0.75 each. "
    "If I buy 6 apples and 4 oranges and pay with a $20 bill, how much change do I get?"
)

direct = ChatPromptTemplate.from_messages([
    ("system", "Answer with just the final numeric answer in dollars. No explanation."),
    ("human", "{q}"),
])

cot = ChatPromptTemplate.from_messages([
    ("system", "Solve the problem. First think step by step, then give the final answer on a last line starting with 'Answer:'."),
    ("human", "{q}"),
])

print("--- Direct ---")
print((direct | llm).invoke({"q": question}).content)

print("\n--- Chain-of-thought ---")
print((cot | llm).invoke({"q": question}).content)

**TODO**: Invent 3 harder word problems (multi-step, trap-laden) and compare direct vs CoT answers. Which is more reliable?

## Part 3: Role / Persona Prompting

Build a "senior Python code reviewer" that returns issues.

In [ ]:
reviewer = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior Python code reviewer.\n"
     "You prioritize correctness, readability, and security.\n"
     "You NEVER approve code with hardcoded secrets.\n"
     "Return issues as a JSON array of objects with keys: "
     "'line' (int), 'severity' ('low'|'med'|'high'), 'message' (str). "
     "Return ONLY the JSON, no prose."),
    ("human", "Review this code:\n\n```python\n{code}\n```"),
])

bad_code = '''
API_KEY = "sk-ABC123SECRET"

def fetch(user):
    q = "SELECT * FROM users WHERE name='" + user + "'"
    return db.execute(q)

def total(items):
    return sum([i for i in items])
'''

print((reviewer | llm).invoke({"code": bad_code}).content)

**TODO**: Change the persona (e.g. a strict security auditor, or a friendly mentor) and observe how tone and priorities shift on the SAME code.

## Part 4: Structured Output with Pydantic

Prose responses are brittle. Use `with_structured_output` for guaranteed schemas.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Ticket(BaseModel):
    intent: Literal["billing", "technical", "sales", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(..., description="<= 20 words")

structured_llm = llm.with_structured_output(Ticket)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract a ticket from the user's message."),
    ("human", "{message}"),
])

for msg, _ in test_messages[:3]:
    chain = prompt | structured_llm
    result = chain.invoke({"message": msg})
    print(result)

**TODO**: Add a `confidence: float` field (0-1) to the `Ticket` model and a `category_reason: str` explaining the category. Re-run.

## Part 5: Prompt Evaluation Harness

Treat prompts like code: evaluate versions on a fixed test set.

In [ ]:
import time

def run_suite(prompt, cases, name):
    chain = prompt | llm
    start = time.time()
    correct = 0
    for msg, expected in cases:
        out = chain.invoke({"message": msg}).content.strip().lower()
        correct += int(out == expected)
    dt = time.time() - start
    acc = correct / len(cases)
    print(f"{name:<20} acc={acc:.1%} time={dt:.1f}s")
    return {"name": name, "accuracy": acc, "time_s": dt}

results = [
    run_suite(zero_shot, test_messages, "zero_shot_v1"),
    run_suite(few_shot, test_messages, "few_shot_v1"),
]
results

**TODO (challenge)**:

1. Write a THIRD prompt version that tries to beat both (e.g. few-shot + explicit reasoning step + confidence).
2. Add at least 10 more test cases.
3. Print a ranked comparison table (accuracy, latency, tokens if you can).
4. Commit the winning prompt to a `prompts/classify_ticket.md` file with a version number.

## Reflection

- When was zero-shot enough? When did few-shot clearly win?
- Did CoT ever give you a WORSE answer? Why might that happen?
- Which part of prompt engineering felt most like software engineering?
- What would you log to catch prompt regressions in production?